<a href="https://colab.research.google.com/github/sanmeshh/pytorch_learning/blob/8.ANN_optimized/ANN_fashion_optimized_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset,DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device:{device}')

device:cpu


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
#reproducibility
torch.manual_seed(108)

In [6]:
df=pd.read_csv('/content/drive/MyDrive/fashion-mnist_train.csv')

In [7]:
X_train=df.iloc[:,1:].values
y_train=df.iloc[:,0].values

In [8]:
df2=pd.read_csv('/content/drive/MyDrive/fashion-mnist_test.csv')
X_test=df2.iloc[:,1:].values
y_test=df2.iloc[:,0].values


In [9]:
#scaling the features values are like 0,142,213
X_train=X_train/255.0
X_test=X_test/255.0

In [10]:
X_train.shape

(60000, 784)

In [11]:
#custom dataset
class CustomDataset(Dataset):
  def __init__(self,features,labels):
    self.features=torch.tensor(features,dtype=torch.float32)#remember float for features
    self.labels=torch.tensor(labels,dtype=torch.long)#and long for labels
  def __len__(self):
    return len(self.features)
  def __getitem__(self,index):
    return self.features[index],self.labels[index]


In [12]:
#create train_dataset object
train_dataset=CustomDataset(X_train,y_train)


In [13]:
len(train_dataset)

60000

In [14]:
train_dataset[3]

(tensor([0.0000, 0.0000, 0.0000, 0.0039, 0.0078, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.4471, 0.7176, 0.4392, 0.2157, 0.0902, 0.2824, 0.4000, 0.6471,
         0.6275, 0.1098, 0.0000, 0.0000, 0.0000, 0.0039, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0039, 0.0000, 0.0000, 0.0941,
         0.7373, 0.6392, 0.3647, 0.5333, 0.6000, 0.6588, 0.9882, 0.6824, 0.5333,
         0.6510, 0.5098, 0.4824, 0.5137, 0.2588, 0.0000, 0.0000, 0.0039, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0078, 0.0000, 0.0392, 0.6157,
         0.8471, 0.8863, 0.8157, 0.5569, 0.2588, 0.4510, 0.5843, 0.9020, 0.7451,
         0.7686, 0.7765, 0.6745, 0.8706, 0.4196, 0.6471, 0.8275, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.4627,
         0.8392, 0.6824, 0.6588, 0.4275, 0.7843, 0.4863, 0.5882, 0.5608, 0.2275,
         0.2471, 0.3490, 0.5373, 0.3804, 0.6588, 0.5412, 0.5569, 0.7647, 0.6118,
         0.0000, 0.0000, 0.0

In [15]:
test_dataset=CustomDataset(X_test,y_test)

In [16]:
#dividing data into batches
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=False,pin_memory=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False,pin_memory=True)

In [18]:
#define NN class

class MyNN(nn.Module):
  def __init__(self,num_features):

    super().__init__()#imp step
    self.model=nn.Sequential(
        nn.Linear(num_features,128),#num_features=784
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(p=0.3),#add dropout after activation and 0.3 means 30% are turn off
        nn.Linear(128,64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(64,10)
        )#4 layers 784->128->64->10

  def forward(self,x):
    return self.model(x)


In [19]:
#set learning rate and epochs
epochs=100
lr=0.1

In [20]:
#model instance
model=MyNN(X_train.shape[1])
model=model.to(device)
#loss func
criterion=nn.CrossEntropyLoss()#take cares of the softmax as well

#optimizer                                   #regularization
optimizer=optim.SGD(model.parameters(),lr=lr,weight_decay=1e-4)

In [21]:
#training loop
for epoch in range(epochs):
  TotalEpochLoss=0
  for batch_features,batch_labels in train_loader:

    #move data to gpu
    batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)

    #forward pass
    outputs=model(batch_features)

    #loss
    loss=criterion(outputs,batch_labels)

    #backward pass
    optimizer.zero_grad()
    loss.backward()


    #update grads
    optimizer.step()

    #calc total loss of each epoch
    TotalEpochLoss+=loss.item()

  avg_loss=TotalEpochLoss/len(train_loader)

  print(f'Epoch:{epoch+1},Loss:{avg_loss}')







Epoch:1,Loss:0.6024459536830584
Epoch:2,Loss:0.4760952557007472
Epoch:3,Loss:0.4371453219453494
Epoch:4,Loss:0.4176731714606285
Epoch:5,Loss:0.39902944658994677
Epoch:6,Loss:0.3856469426592191
Epoch:7,Loss:0.3752590047955513
Epoch:8,Loss:0.36596220991611483
Epoch:9,Loss:0.35500590526660286
Epoch:10,Loss:0.3482272504091263
Epoch:11,Loss:0.34444157996575037
Epoch:12,Loss:0.3379646037300428
Epoch:13,Loss:0.3335512491683165
Epoch:14,Loss:0.33225572168628376
Epoch:15,Loss:0.32440448992053666
Epoch:16,Loss:0.31785570095380145
Epoch:17,Loss:0.31315376177628834
Epoch:18,Loss:0.3106059324045976
Epoch:19,Loss:0.3088333997031053
Epoch:20,Loss:0.30282417110006016
Epoch:21,Loss:0.3051467864831289
Epoch:22,Loss:0.29661386328140893
Epoch:23,Loss:0.2942339809169372
Epoch:24,Loss:0.2921149703741074
Epoch:25,Loss:0.291412333915631
Epoch:26,Loss:0.28968175891637804
Epoch:27,Loss:0.2878419133086999
Epoch:28,Loss:0.2880967865566413
Epoch:29,Loss:0.2787628928244114
Epoch:30,Loss:0.2800186298350493
Epoch:31,

In [22]:
# set model to evaluation mode
model.eval()

MyNN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [23]:
#evaluation code
total=0
correct=0

#we dont want to calculate gradients during evaluation
with torch.no_grad():

  for batch_features,batch_labels in test_loader:
    batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)

    outputs=model(batch_features)


    _ , predicted=torch.max(outputs,1)

    #getting the total no of samples by adding samples of each batch
    total=total+batch_labels.shape[0]
    # print(total)

    correct+=(predicted==batch_labels).sum().item()

  print(correct/total)








0.8897


In [24]:
#evaluation code
total=0
correct=0

#we dont want to calculate gradients during evaluation
with torch.no_grad():

  for batch_features,batch_labels in train_loader:
    batch_features,batch_labels=batch_features.to(device),batch_labels.to(device)

    outputs=model(batch_features)


    _ , predicted=torch.max(outputs,1)

    #getting the total no of samples by adding samples of each batch
    total=total+batch_labels.shape[0]
    # print(total)

    correct+=(predicted==batch_labels).sum().item()

  print(correct/total)

0.9351666666666667


In [ ]:
#as you can see we have lower our training accuracy from 98 to 93% since our test accuracy was 88% and huge diff bw test and training accuracy indicated overfitting.
#take care hare krishna